<a href="https://colab.research.google.com/github/Hunterashcraft04/SemanticsOfDataViz_HAshcraftGTUgThesis/blob/main/DataVizThesis_Script/Pilot%20Data/taskMetricsPilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# imports
import os
import shutil
import pandas as pd
import numpy as np
from google.colab import files
from pathlib import Path
from statsmodels.stats.anova import AnovaRM
from statsmodels.stats.weightstats import DescrStatsW


In [7]:
# upload data
ulfs = files.upload()

for filename in ulfs.keys():
    if not filename.lower().endswith(('.csv', '.xlsx')):
        raise Exception(f"Invalid file type: {filename}")

cDir = "uploaded_data"

if os.path.exists(cDir):
    shutil.rmtree(cDir)
os.makedirs(cDir, exist_ok=True)

for filename, content in ulfs.items():
    fnsPath = os.path.join(cDir, filename)
    with open(fnsPath, 'wb') as f:
        f.write(content)

ulfsPath = sorted([
    str(p) for p in Path(cDir).glob('**/*')
    if p.suffix.lower() in ('.csv', '.xlsx')
])

print(f"Uploaded and saved {len(ulfsPath)} file(s).")
for path in ulfsPath:
    print(f" - {path}")


Saving pilot3_run1.csv to pilot3_run1.csv
Saving pilot4_run1.csv to pilot4_run1.csv
Saving pilot5_run1.csv to pilot5_run1.csv
Saving pilot6_run1.csv to pilot6_run1.csv
Saving pilot7_run1.csv to pilot7_run1.csv
Saving test1_run1.csv to test1_run1.csv
Saving test2_run1.csv to test2_run1.csv
Saving test3_run1.csv to test3_run1.csv
Saving pilot1_run1.csv to pilot1_run1.csv
Saving pilot2_run1.csv to pilot2_run1.csv
Uploaded and saved 10 file(s).
 - uploaded_data/pilot1_run1.csv
 - uploaded_data/pilot2_run1.csv
 - uploaded_data/pilot3_run1.csv
 - uploaded_data/pilot4_run1.csv
 - uploaded_data/pilot5_run1.csv
 - uploaded_data/pilot6_run1.csv
 - uploaded_data/pilot7_run1.csv
 - uploaded_data/test1_run1.csv
 - uploaded_data/test2_run1.csv
 - uploaded_data/test3_run1.csv


In [8]:
# metrics; collects RT and incorrect answers
all_task_summaries = []
incorrect_trials_list = []

print("Processing files...\n")

for path in ulfsPath:
    filename = os.path.basename(path)

    if path.lower().endswith('.csv'):
        with open(path, 'r') as f:
            lines = [f.readline() for _ in range(10)]
        header_idx = next((i for i, l in enumerate(lines) if 'TrialCode' in l or 'Trial' in l), 0)
        df = pd.read_csv(path, skiprows=header_idx)
    else:
        # (.xlsx) files
        df_temp = pd.read_excel(path, nrows=10)
        header_idx = next((i for i, row in enumerate(df_temp.values) if any('TrialCode' in str(x) for x in row)), None)
        df = pd.read_excel(path, skiprows=header_idx + 1 if header_idx is not None else 0)

    # clean the data
    df.columns = df.columns.str.strip()

    # ensure necessary columns are present
    required_cols = ['TrialCode', 'Stimulus', 'RT', 'Correct', 'r', 'z']
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        print(f"Skipping {filename}: Missing columns {missing_cols}")
        continue

    df['RT'] = pd.to_numeric(df['RT'], errors='coerce')
    df['Correct'] = pd.to_numeric(df['Correct'], errors='coerce')
    df['r'] = pd.to_numeric(df['r'], errors='coerce')
    df['z'] = pd.to_numeric(df['z'], errors='coerce')
    df['TrialCode'] = df['TrialCode'].astype(str).str.strip()
    df['Stimulus'] = df['Stimulus'].astype(str).str.strip()

    df = df[df['TrialCode'].notna() & (df['TrialCode'] != '')]

    # task specific metrics (RT and accuracy)
    task_summary = df.groupby('TrialCode').agg(
        Avg_RT=('RT', 'mean'),
        Accuracy_Pct=('Correct', lambda x: x.mean() * 100)
    ).reset_index()

    task_summary['Source_File'] = filename
    all_task_summaries.append(task_summary)

    # correct incorrect
    incorrect_df = df[df['Correct'] == 0].copy()
    if not incorrect_df.empty:
        incorrect_df['Source_File'] = filename
        incorrect_df = incorrect_df[['Source_File', 'Stimulus', 'TrialCode', 'r', 'z', 'RT']]
        incorrect_trials_list.append(incorrect_df)

# Combine results from all files
if all_task_summaries:
    final_metrics = pd.concat(all_task_summaries, ignore_index=True)
    final_metrics = final_metrics.groupby('TrialCode').mean(numeric_only=True).reset_index()

    print("Task accuracy and RT")
    final_metrics['Accuracy_Pct'] = final_metrics['Accuracy_Pct'].map('{:.2f}%'.format)
    final_metrics['Avg_RT'] = final_metrics['Avg_RT'].map('{:.4f}s'.format)
    print(final_metrics.to_string(index=False))
else:
    print("No valid task metrics compiled")

if incorrect_trials_list:
    final_incorrect = pd.concat(incorrect_trials_list, ignore_index=True)
    print("Incorrect Trials")
    print(final_incorrect.to_string(index=False))


Processing files...

Task accuracy and RT
   TrialCode  Avg_RT Accuracy_Pct
Movement (M) 1.5377s       90.56%
 Outlier (O) 1.1021s       87.22%
 Passive (P) 0.4783s         nan%
Question (Q) 1.0151s       92.22%
         nan    nans         nan%
Incorrect Trials
    Source_File                                                      Stimulus    TrialCode      r     z       RT
pilot1_run1.csv   scatter_plot7_posSrngCor(r=0.819)_out(MD=4.476,z=4.909).png  Outlier (O)  0.819 4.909 1.803860
pilot1_run1.csv   scatter_plot5_posSrngCor(r=0.879)_out(MD=4.432,z=4.984).png  Outlier (O)  0.879 4.984 1.342690
pilot1_run1.csv     scatter_plot41_negCor(r=-0.109)_out(MD=3.868,z=4.046).png  Outlier (O) -0.109 4.046 1.184107
pilot1_run1.csv     scatter_plot37_negCor(r=-0.038)_out(MD=4.032,z=4.057).png Question (Q) -0.038 4.057 0.891870
pilot1_run1.csv scatter_plot67_negSrngCor(r=-0.866)_out(MD=3.966,z=4.075).png  Outlier (O) -0.866 4.075 0.766981
pilot1_run1.csv                       scatter_plot68_negSrn

In [12]:
# Active tasks: Movement, Outlier, Question
tasks = ['Movement (M)', 'Outlier (O)', 'Question (Q)']

# Use participant-level data generated by the existing metrics cell
summary = pd.concat(all_task_summaries, ignore_index=True)
summary = summary.rename(columns={'Source_File': 'Participant'})

for label, variable in [
    ('RT', 'Avg_RT'),
    ('Accuracy', 'Accuracy_Pct')
]:
    df = summary[
        summary['TrialCode'].isin(tasks)
    ].dropna(subset=[variable]).copy()

    df = df.groupby('Participant').filter(
        lambda x: x['TrialCode'].nunique() == len(tasks)
    )

    # RM-ANOVA
    print(f"\n{'=' * 50}")
    print(f"{label} RM-ANOVA (M, O, Q)")
    print(f"Participants: {df['Participant'].nunique()}")

    print(AnovaRM(
        data=df,
        depvar=variable,
        subject='Participant',
        within=['TrialCode']
    ).fit())

    # Participant × task format
    wide = df.pivot(
        index='Participant',
        columns='TrialCode',
        values=variable
    )

    # Paired t-tests with Bonferroni correction
    print(f"\nPaired t-tests ({label})")

    for i in range(len(tasks)):
        for j in range(i + 1, len(tasks)):
            a, b = tasks[i], tasks[j]

            differences = (wide[a] - wide[b]).dropna()

            t, p, dfree = DescrStatsW(differences).ttest_mean(0)

            p_bonf = min(p * 3, 1.0)

            print(
                f"{a} vs {b}: "
                f"t({dfree:.0f}) = {t:.4f}, "
                f"p = {p:.4f}, "
                f"p_Bonf = {p_bonf:.4f}"
            )


RT RM-ANOVA (M, O, Q)
Participants: 10
                 Anova
          F Value Num DF  Den DF Pr > F
---------------------------------------
TrialCode 14.0741 2.0000 18.0000 0.0002


Paired t-tests (RT)
Movement (M) vs Outlier (O): t(9) = 3.8034, p = 0.0042, p_Bonf = 0.0126
Movement (M) vs Question (Q): t(9) = 4.0590, p = 0.0028, p_Bonf = 0.0085
Outlier (O) vs Question (Q): t(9) = 1.4243, p = 0.1881, p_Bonf = 0.5643

Accuracy RM-ANOVA (M, O, Q)
Participants: 10
                 Anova
          F Value Num DF  Den DF Pr > F
---------------------------------------
TrialCode  0.5216 2.0000 18.0000 0.6023


Paired t-tests (Accuracy)
Movement (M) vs Outlier (O): t(9) = 0.5859, p = 0.5724, p_Bonf = 1.0000
Movement (M) vs Question (Q): t(9) = -0.4192, p = 0.6849, p_Bonf = 1.0000
Outlier (O) vs Question (Q): t(9) = -0.9736, p = 0.3557, p_Bonf = 1.0000
